In [0]:
# Databricks notebook source
# ETL - Squad 3 - ecommerce_clientes
# Fluxo: Raw CSV -> Bronze Delta -> Silver Delta -> Gold Delta/SQL Server

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
SOURCE_FILE = "ecommerce_clientes.csv"
SOURCE_PATH = f"{RAW_BATCH_PATH}{SOURCE_FILE}"

CSV_OPTIONS = {
    "header": "true",
    "inferSchema": "false",
}

EXPECTED_COLUMNS = [
    "id_cliente",
    "uuid_cliente",
    "nome",
    "sobrenome",
    "email",
    "senha_hash",
    "dt_cadastro",
    "dt_ultima_atualizacao",
]

KEY_COLUMNS = ["id_cliente"]

BRONZE_TABLE = "ecommerce_clientes"
BRONZE_PATH = f"{BRONZE_BASE_PATH}{BRONZE_TABLE}"

PARTITION_DATE_COLUMN = "dt_cadastro"
BRONZE_WRITE_MODE = "overwrite"

print("SOURCE_PATH:", SOURCE_PATH)
print("BRONZE_PATH:", BRONZE_PATH)
print("PARTITION_DATE_COLUMN:", PARTITION_DATE_COLUMN)

In [0]:
adls_options = get_adls_options()

print("Opções ADLS configuradas.")

In [0]:

# ler CSV da Raw

df_source = read_source_csv(
    spark=spark,
    source_path=SOURCE_PATH,
    adls_options=adls_options,
    csv_options=CSV_OPTIONS
)

df_source.printSchema()
display(df_source.limit(10))

In [0]:
# contar origem

total_source = df_source.count()

print(f"Total de registros lidos da Raw: {total_source}")

In [0]:
# validar conversão da data de particionamento

from pyspark.sql.functions import col, to_timestamp, count, when

df_test_date = df_source.withColumn(
    "dt_cadastro_convertida",
    to_timestamp(col(PARTITION_DATE_COLUMN))
)

display(
    df_test_date.select(
        count("*").alias("total_linhas"),
        count(when(col(PARTITION_DATE_COLUMN).isNull(), True)).alias("dt_cadastro_nula_origem"),
        count(
            when(
                col(PARTITION_DATE_COLUMN).isNotNull() &
                col("dt_cadastro_convertida").isNull(),
                True
            )
        ).alias("falhas_conversao")
    )
)

In [0]:

# criar DataFrame Bronze

from pyspark.sql.functions import current_timestamp, lit, year, month

df_bronze = (
    df_source
    .withColumn("bronze_ingested_at", current_timestamp())
    .withColumn("bronze_source_file", lit(SOURCE_PATH))
    .withColumn("_partition_date", to_timestamp(col(PARTITION_DATE_COLUMN)))
    .withColumn("ano", year(col("_partition_date")))
    .withColumn("mes", month(col("_partition_date")))
    .drop("_partition_date")
)

In [0]:
# visualizar Bronze antes de gravar

display(
    df_bronze
    .select(
        "id_cliente",
        "dt_cadastro",
        "ano",
        "mes",
        "bronze_ingested_at",
        "bronze_source_file"
    )
    .limit(20)
)

In [0]:
# validar particionamento Bronze
display(
    df_bronze.select(
        count("*").alias("total_linhas"),
        count(when(col("ano").isNull(), True)).alias("ano_nulo"),
        count(when(col("mes").isNull(), True)).alias("mes_nulo")
    )
)

In [0]:
# gravar Bronze Delta

(
    df_bronze
    .write
    .format("delta")
    .options(**adls_options)
    .mode(BRONZE_WRITE_MODE)
    .partitionBy("ano", "mes")
    .save(BRONZE_PATH)
)

print(f"Dados gravados com sucesso na Bronze: {BRONZE_PATH}")

In [0]:
# ler Bronze gravada

df_bronze_saved = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(BRONZE_PATH)
)

df_bronze_saved.printSchema()

total_bronze = df_bronze_saved.count()

print(f"Total de registros na Bronze: {total_bronze}")

display(df_bronze_saved.limit(10))

In [0]:
# validar origem x Bronze

print(f"Total origem Raw: {total_source}")
print(f"Total gravado Bronze: {total_bronze}")

if total_source == total_bronze:
    print("Validação Bronze OK: quantidade de registros da Raw e da Bronze é igual.")
else:
    print("Atenção: quantidade de registros diferente entre Raw e Bronze.")